Predictive RNN Gridworld Experiment
Goal: Train an RNN to predict o_t+1 from o_t and a_t, and examine whetehr its hidden state encodes spatial position.

In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Imports

In [13]:
from gridworld import GridWorld, Agent
import sys
import numpy as np
import pandas as pd
import torch
from torch import nn

print(sys.executable)
print(torch.__version__)

/home/seblee/miniconda3/bin/python
2.14.0+cu130


State variables

In [14]:
HEIGHT = 20
WIDTH = 25
VIEW_SIZE = 5

START_X = 6
START_Y = 6
START_DIRECTION = 2

N_STEPS = 50

ACTION_PROBS = [0.6, 0.15, 0.15, 0.1]

SEED = 42

Gridworld Creation

In [15]:
grid = GridWorld(HEIGHT, WIDTH)
grid.add_rectangle_region(5, 10, 5, 20)
grid.add_rectangle_region(10, 15, 10, 15)
grid.add_square_shape(6, 6, 3, 7)
grid.add_triangle_shape(6, 11, 3, 0, 6)
grid.add_single_shape(7, 17, 5)
grid.add_single_shape(6, 16, 5)
grid.add_single_shape(6, 18, 5)
grid.add_single_shape(8, 16, 5)
grid.add_single_shape(8, 18, 5)
grid.add_rectangle_shape(11, 12, 3, 1, 3)
grid.add_rectangle_shape(12, 11, 1, 3, 4)


Agent Creation

In [16]:

agent = Agent(START_X, START_Y, VIEW_SIZE, grid, START_DIRECTION)

Random Walk

In [17]:
rng = np.random.default_rng(SEED)

for i in range(500):
    action = rng.choice([0, 1, 2, 3], p=[0.6, 0.15, 0.15, 0.1])

    if action == 0:
        agent.move()
        print(f"moved towards {agent.direction}")
    elif action == 3:
        agent.stop()
        print("stopped")
    else:
        
        agent.rotate(action)
        print(f"rotated towards {agent.direction}")
    observation = agent.observe()
    #print(pd.DataFrame(observation))

print(agent.states)
print(len(agent.states))
print(len(agent.observations))
print(len(agent.actions))

rotated towards 3
moved towards 3
rotated towards 0
rotated towards 3
moved towards 3
stopped
rotated towards 0
rotated towards 1
moved towards 1
moved towards 1
moved towards 1
stopped
rotated towards 0
rotated towards 1
moved towards 1
moved towards 1
moved towards 1
moved towards 1
rotated towards 2
rotated towards 1
rotated towards 2
moved towards 2
stopped
rotated towards 3
rotated towards 0
moved towards 0
moved towards 0
moved towards 0
moved towards 0
rotated towards 3
rotated towards 2
stopped
moved towards 2
moved towards 2
moved towards 2
moved towards 2
moved towards 2
moved towards 2
moved towards 2
rotated towards 1
moved towards 1
rotated towards 2
rotated towards 1
moved towards 1
rotated towards 2
rotated towards 3
moved towards 3
moved towards 3
rotated towards 2
moved towards 2
moved towards 2
moved towards 2
rotated towards 3
rotated towards 2
rotated towards 1
rotated towards 2
moved towards 2
moved towards 2
moved towards 2
moved towards 2
rotated towards 1
moved 

In [18]:
observations = torch.tensor(
    np.array(agent.observations),
    dtype = torch.float32
)

actions = torch.tensor(
    agent.actions, 
    dtype = torch.long
)

states = torch.tensor(
    agent.states,
    dtype = torch.long
)

print(observations.shape)
print(actions.shape)
print(states.shape)

observations = observations.flatten(start_dim = 1)
print(observations.shape)

actions_onehot = torch.nn.functional.one_hot(
    actions,
    num_classes = 4
).float()

print(actions_onehot.shape)

input_obs = observations[:-1]
target_obs = observations[1:]

inputs = torch.cat(
    (input_obs, actions_onehot),
    dim = 1
)

print("input observations: ", input_obs.shape)
print("actions: ", actions_onehot.shape)
print("RNN inputs: ", inputs.shape)
print("targets: ", target_obs.shape)

torch.Size([501, 5, 5])
torch.Size([500])
torch.Size([501, 3])
torch.Size([501, 25])
torch.Size([500, 4])
input observations:  torch.Size([500, 25])
actions:  torch.Size([500, 4])
RNN inputs:  torch.Size([500, 29])
targets:  torch.Size([500, 25])
